In [2]:
from google.colab import drive
import sys
import os

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Definer projektstien
PROJECT_ROOT = "/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep"
SRC_DIR = os.path.join(PROJECT_ROOT, "src")

# 3. Tilføj stien til Pythons søgesti (det er dette trin, der fjerner fejlen)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

os.makedirs(SRC_DIR, exist_ok=True)
print(f"Projektsti tilføjet! src findes i: {SRC_DIR}")

Mounted at /content/drive
Projektsti tilføjet! src findes i: /content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src


In [3]:
%%writefile "/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/__init__.py"
# Marks src as a Python package

Overwriting /content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/__init__.py


In [3]:
%%writefile "/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/dataset.py"
import torch
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import zipfile
import json
import shutil
from pathlib import Path

# Automatic Extraction Setup
PROJECT_DIR = Path('/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep')
ZIP_PATH = PROJECT_DIR / 'plant_data.zip'
MANIFEST_PATH = PROJECT_DIR / 'splits.json'

DATA_ROOT = Path('/content/data')
BASE_DIR = DATA_ROOT / 'PlantVillage'
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'val'
TEST_DIR = DATA_ROOT / 'test'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def ensure_data_extracted():
    """Extracts zip dataset to local disk if not already present."""
    if not all(d.exists() and any(d.iterdir()) for d in (TRAIN_DIR, VAL_DIR, TEST_DIR)):
        print("Data ikke fundet lokalt. Udpakker plant_data.zip til /content/data/ ...")

        if not BASE_DIR.exists():
            with zipfile.ZipFile(ZIP_PATH, 'r') as z:
                z.extractall(DATA_ROOT)

        with open(MANIFEST_PATH) as f:
            split = json.load(f)

        for split_name, target_dir in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
            for rel_path in split[split_name]:
                src = BASE_DIR / rel_path
                dst = target_dir / rel_path
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(str(src), str(dst))

        print("Data udpakning færdig!")

def get_transforms(exp_id: str, img_size: int = 224):
    if exp_id == "E2":
        train_transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            T.RandomRotation(15),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        train_transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    eval_transform = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    return train_transform, eval_transform

def build_dataloaders(train_dir: str, val_dir: str, test_dir: str, config: dict):
    # Ensure dataset is on fast local disk before loading
    ensure_data_extracted()

    train_tf, eval_tf = get_transforms(config["experiment_id"], config["img_size"])

    train_dataset = ImageFolder(train_dir, transform=train_tf)
    val_dataset = ImageFolder(val_dir, transform=eval_tf)
    test_dataset = ImageFolder(test_dir, transform=eval_tf)

    sampler = None
    shuffle = True

    if config["experiment_id"] == "E1c":
        class_counts = np.bincount([label for _, label in train_dataset.samples])
        class_weights = 1.0 / class_counts
        sample_weights = [class_weights[label] for _, label in train_dataset.samples]
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        shuffle = False

    train_loader = DataLoader(
        train_dataset, batch_size=config["batch_size"], shuffle=shuffle, sampler=sampler, num_workers=2
    )
    val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader, train_dataset.classes

Overwriting /content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/dataset.py


In [4]:
%%writefile "/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/losses.py"
import torch
import torch.nn as nn
import numpy as np

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

def get_loss_function(exp_id: str, train_loader=None, device="cuda"):
    if exp_id == "E1a":
        targets = [label for _, label in train_loader.dataset.samples]
        counts = np.bincount(targets)
        weights = 1.0 / counts
        weights = torch.tensor(weights / weights.sum(), dtype=torch.float32).to(device)
        return nn.CrossEntropyLoss(weight=weights)
    elif exp_id == "E1b":
        return FocalLoss(gamma=2.0)
    else:
        return nn.CrossEntropyLoss()

Writing /content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/losses.py


In [5]:
%%writefile "/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/models.py"
import torch.nn as nn
import torchvision.models as models

def get_resnet18(num_classes: int = 15):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

Writing /content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/models.py


In [3]:
%%writefile "/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/trainer.py"
import torch
from sklearn.metrics import classification_report, f1_score, accuracy_score

def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    top1 = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    report = classification_report(all_targets, all_preds, output_dict=True, zero_division=0)

    return top1, macro_f1, report

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    return running_loss / len(dataloader.dataset)

Overwriting /content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep/src/trainer.py
